# NLP 下游任务（DeltaNet）

本 Notebook 说明在本样例目录布局下，如何在 **昇腾 NPU** 上完成 **GPT-2 风格**自回归语言模型的训练与 `lm_eval` 评测；注意力侧采用 **DeltaNet（Delta Rule）** 替换路径（见 `training/attentions/`）。

---

## 1. 模型架构、训练设定与实验环境准备

以下路径均以 **样例根目录**（与顶层 `README.md` 同级，即 `ascend_tla_deltanet/`）为当前工作目录。

NLP 侧采用 **GPT-2 风格的自回归语言模型架构** 进行训练，配置文件位于：

- `internal/tla/nlp/tla-nlp-frame/training/democonfig/config.json`

训练数据采用 **SlimPajama**，并在 Ascend NPU 环境下通过 `train.sh` 进行分布式训练；
随后使用 `lm_eval` 对 PIQA、HellaSwag、WinoGrande、ARC-E、ARC-C 等任务做统一评估。

### 1.1 实验环境准备

在开始训练和评测前，先完成 NLP 运行环境配置。以下命令可直接参考执行（路径按实际环境修改）。

#### Step 1. 安装基础依赖

```bash
pip install transformers==4.57.3
pip install datasets accelerate deepspeed
```

#### Step 2. 安装评测工具（lm-eval）

```bash
pip install lm-eval
```

#### Step 3. 下载 SlimPajama 数据集

```bash
python internal/tla/nlp/tla-nlp-frame/preparation/download_data.py
```

#### Step 4. 处理 SlimPajama（分词与样本打包）

```bash
python internal/tla/nlp/tla-nlp-frame/preparation/tokenizer_data.py \
  --dataset <downloaded-dataset-path> \
  --split train \
  --ctx_len 4096 \
  --seq_len 4096 \
  --tokenizer internal/tla/nlp/tla-nlp-frame/training/democonfig \
  --output <tokenized-output-path> \
  --num_proc 32
```

参数说明：

- `<downloaded-dataset-path>`：下载后的 SlimPajama 数据路径。
- `--ctx_len` / `--seq_len`：上下文与序列长度（示例为 `4096`）。
- `--tokenizer`：Tokenizer 配置目录，保持与训练配置一致。
- `<tokenized-output-path>`：分词后数据输出路径。
- `--num_proc`：并行处理进程数，建议按机器资源调整。

#### Step 5. （可选）快速验证评测命令格式

```bash
lm_eval --model hf \
  --model_args pretrained=<ckpt-path>,dtype="bfloat16" \
  --tasks piqa,hellaswag,winogrande,arc_easy,arc_challenge \
  --batch_size 64 \
  --device npu:0 \
  --show_config
```

完成以上环境准备后，再执行本 Notebook 第 3 节中的训练与下游评测流程。

---

## 2. NLP 下游任务测试命令

### 2.1 数据准备（SlimPajama）

```bash
python internal/tla/nlp/tla-nlp-frame/preparation/download_data.py

python internal/tla/nlp/tla-nlp-frame/preparation/tokenizer_data.py \
  --dataset <downloaded-dataset-path> \
  --split train \
  --ctx_len 4096 \
  --seq_len 4096 \
  --tokenizer internal/tla/nlp/tla-nlp-frame/training/democonfig \
  --output <tokenized-output-path> \
  --num_proc 32
```

### 2.2 训练命令

```bash
bash internal/tla/nlp/tla-nlp-frame/training/train.sh \
  lr=3e-4 scheduler=cosine_with_min_lr \
  batch=4 update=4 warmup=1024 steps=30720 \
  context=4096 gpus=8 nodes=1 \
  path=<output-path> name=<exp-name> \
  model=internal/tla/nlp/tla-nlp-frame/training/democonfig/config.json \
  tokenizer=internal/tla/nlp/tla-nlp-frame/training/democonfig \
  data=SlimPajama cache=<tokenized-dataset-path>
```

---

## 3. 评测对象与实验数据

在 **GPT-2 style architecture** 上完成训练，并对下游任务进行评估。下面给出对应实验数据。

任务简要说明：

- **PIQA**：物理常识推理任务，测试模型对日常物理世界知识的理解。
- **HellaSwag**：常识与情景续写选择任务，测试语境理解和合理续写能力。
- **WinoGrande**：代词消解任务，测试模型在歧义语句中的指代推理能力。
- **ARC-E**：AI2 科学题（Easy），偏基础科学常识与简单推理。
- **ARC-C**：AI2 科学题（Challenge），难度更高，更依赖多步推理能力。

### 3.1 DeltaNet（本样例）

| Backend | PIQA (acc) | HellaSwag (acc_n) | WinoGrande (acc) | Arc_e (acc) | Arc_c (acc_n) |
|:--------|-----------:|------------------:|-----------------:|------------:|--------------:|
| CUDA | 63.8 | 32.2 | 52.9 | 46.1 | 20.4 |
| TLA  | 64.3 | 32.7 | 50.4 | 45.0 | 18.9 |

---

## 4. 结论

从当前 NLP 下游评测结果看，在 GPT-2 风格架构上使用 **DeltaNet** 注意力时，Ascend-TLA 与 CUDA 基线在 PIQA、HellaSwag、ARC-E 等任务上整体处于同一量级；部分指标存在小幅差异，可在上游仓库配置与更长训练 schedule 下进一步对齐。更完整的对比与复现说明见 [Ascend-TLA](https://gitcode.com/SMULL_Group/Ascend-TLA.git)。